In [22]:
# ============================================================
# CELL 1: Load only first 50k polygons from parks.tsv
# ============================================================

import numpy as np
from shapely import wkt as shapely_wkt
from collections import Counter

DATA_PATH = "/raid/ruban/data/parks.tsv"
POLY_COUNT = 50000

def _parse_tsv_line(line: str):
    parts = line.rstrip("\n").split("\t")
    if len(parts) < 2:
        raise ValueError("bad tsv line")
    feat_id = int(parts[0])
    wkt_str = parts[1]
    return feat_id, wkt_str

def _linestring_to_polygon(geom):
    from shapely.geometry import Polygon
    coords = list(geom.coords)
    if len(coords) == 0:
        raise ValueError("empty geometry")
    if len(coords) == 2:
        raise ValueError("LineString too short for polygon conversion: 2 coords")
    if coords[0] != coords[-1]:
        coords.append(coords[0])
    if len(coords) < 4:
        raise ValueError(f"closed ring too short: {len(coords)} coords")
    return Polygon(coords)

def _geom_to_polygon(geom):
    if geom.is_empty:
        raise ValueError("empty geometry")

    gtype = geom.geom_type

    if gtype == "Polygon":
        return geom

    if gtype == "MultiPolygon":
        parts = list(geom.geoms)
        if not parts:
            raise ValueError("empty multipolygon")
        return max(parts, key=lambda g: g.area)

    if gtype == "LineString":
        return _linestring_to_polygon(geom)

    raise ValueError(f"unsupported geometry type: {gtype}")

geometries = []
poly_ids = []
skip_reasons = Counter()

with open(DATA_PATH, "r", encoding="utf-8") as fh:
    for lineno, line in enumerate(fh):
        if len(geometries) >= POLY_COUNT:
            break

        try:
            feat_id, wkt_str = _parse_tsv_line(line)
            geom = shapely_wkt.loads(wkt_str)
            geom = _geom_to_polygon(geom)

            if not geom.is_valid:
                geom = geom.buffer(0)

            if geom.is_empty:
                skip_reasons["empty_after_fix"] += 1
                continue

            geometries.append(geom)
            poly_ids.append(feat_id)

        except Exception as exc:
            msg = str(exc).lower()
            if "too short" in msg:
                skip_reasons["degenerate_linestring"] += 1
            elif "empty" in msg:
                skip_reasons["empty_geometry"] += 1
            else:
                skip_reasons["other"] += 1

print("Loaded polygons:", len(geometries))
print("Skipped:", sum(skip_reasons.values()))
print("Sample geom type:", geometries[0].geom_type if geometries else None)
print("Sample poly id:", poly_ids[0] if poly_ids else None)

Loaded polygons: 50000
Skipped: 52
Sample geom type: Polygon
Sample poly id: 4061698


In [23]:
# ============================================================
# CELL 2: Normalize rings + build shape descriptors in parallel
# ============================================================

import numpy as np
import multiprocessing as mp

NUM_WORKERS = min(100, mp.cpu_count())
RESAMPLE_PTS = 64   # 64 boundary points -> 128-d descriptor

def _remove_duplicate_last(coords):
    coords = np.asarray(coords, dtype=np.float32)
    if len(coords) >= 2 and np.allclose(coords[0], coords[-1]):
        coords = coords[:-1]
    return coords

def _normalize_ring(coords):
    coords = _remove_duplicate_last(coords)
    if len(coords) < 3:
        return None

    # center
    c = coords.mean(axis=0, keepdims=True)
    coords = coords - c

    # scale to unit max radius
    r = np.linalg.norm(coords, axis=1).max()
    if r <= 1e-12:
        return None
    coords = coords / r

    return coords.astype(np.float32)

def _resample_ring(coords, n_points=64):
    coords = _remove_duplicate_last(coords)
    if coords is None or len(coords) < 3:
        return None

    # close ring explicitly for arc-length resampling
    closed = np.vstack([coords, coords[0]])
    segs = closed[1:] - closed[:-1]
    lens = np.linalg.norm(segs, axis=1)

    total = lens.sum()
    if total <= 1e-12:
        return None

    cum = np.concatenate([[0.0], np.cumsum(lens)])
    targets = np.linspace(0.0, total, n_points, endpoint=False)

    out = []
    j = 0
    for t in targets:
        while j + 1 < len(cum) and cum[j + 1] < t:
            j += 1

        seg_len = lens[j]
        if seg_len <= 1e-12:
            p = closed[j].copy()
        else:
            alpha = (t - cum[j]) / seg_len
            p = closed[j] + alpha * (closed[j + 1] - closed[j])
        out.append(p)

    return np.asarray(out, dtype=np.float32)

def _canonicalize_start(points):
    # start from lexicographically smallest point for stability
    idx = np.lexsort((points[:, 1], points[:, 0]))[0]
    return np.roll(points, -idx, axis=0)

def _descriptor_from_geom(geom):
    try:
        # use exterior ring only for now (fast baseline)
        coords = np.asarray(geom.exterior.coords, dtype=np.float32)
        coords = _normalize_ring(coords)
        if coords is None:
            return None

        pts = _resample_ring(coords, RESAMPLE_PTS)
        if pts is None:
            return None

        pts = _canonicalize_start(pts)

        # flatten to 128-d descriptor
        desc = pts.reshape(-1).astype(np.float32)

        # normalize descriptor
        nrm = np.linalg.norm(desc)
        if nrm <= 1e-12:
            return None
        desc = desc / nrm

        return desc
    except Exception:
        return None

print(f"Building descriptors with {NUM_WORKERS} workers...")

with mp.Pool(processes=NUM_WORKERS) as pool:
    descriptors = pool.map(_descriptor_from_geom, geometries, chunksize=64)

valid_mask = [d is not None for d in descriptors]
valid_indices = [i for i, ok in enumerate(valid_mask) if ok]

descriptors = np.vstack([descriptors[i] for i in valid_indices]).astype(np.float32)
geometries_valid = [geometries[i] for i in valid_indices]
poly_ids_valid = [poly_ids[i] for i in valid_indices]

print("Valid descriptors:", descriptors.shape[0])
print("Descriptor dim   :", descriptors.shape[1])
print("Filtered out     :", POLY_COUNT - len(valid_indices))
print("Sample poly id   :", poly_ids_valid[0])

Building descriptors with 100 workers...
Valid descriptors: 49963
Descriptor dim   : 128
Filtered out     : 37
Sample poly id   : 4061698


In [24]:
# ============================================================
# CELL 3 (FIXED): Build custom shape GT for 1000 queries
# using ORIGINAL dataset indices, not compressed row positions
# ============================================================

import numpy as np

DATA_END_50K = 40000
QUERY_START_50K = 40000
QUERY_END_50K = 50000

NUM_GT_QUERIES = 1000
TOPK_GT = 500

valid_indices_arr = np.asarray(valid_indices, dtype=np.int32)

# rows in descriptor matrix belonging to DB / query split
db_rows = np.where(valid_indices_arr < DATA_END_50K)[0]
query_rows_all = np.where((valid_indices_arr >= QUERY_START_50K) & (valid_indices_arr < QUERY_END_50K))[0]

# take first 1000 valid queries from last 10k
query_rows = query_rows_all[:NUM_GT_QUERIES]

X_db = descriptors[db_rows]         # [num_db_valid, 128]
X_q = descriptors[query_rows]       # [num_q_valid, 128]

# global/original dataset indices
db_global_ids = valid_indices_arr[db_rows]
query_global_ids = valid_indices_arr[query_rows]

print("DB descriptors    :", X_db.shape)
print("Query descriptors :", X_q.shape)
print("DB global range   :", int(db_global_ids.min()), "to", int(db_global_ids.max()))
print("Query global range:", int(query_global_ids.min()), "to", int(query_global_ids.max()))

# cosine similarity because descriptors are normalized
sim = X_q @ X_db.T   # [num_queries, num_db]

# top-k neighbors in DB for each query
topk_idx = np.argpartition(-sim, kth=TOPK_GT-1, axis=1)[:, :TOPK_GT]

# sort those top-k by similarity
row_ids = np.arange(topk_idx.shape[0])[:, None]
topk_scores = sim[row_ids, topk_idx]
order = np.argsort(-topk_scores, axis=1)
topk_sorted = topk_idx[row_ids, order]

# convert local DB row indices -> original dataset indices
custom_gt = {}
for r, qid in enumerate(query_global_ids):
    custom_gt[int(qid)] = db_global_ids[topk_sorted[r]].astype(int).tolist()

print("Custom GT queries:", len(custom_gt))
sample_q = int(query_global_ids[0])
print("Sample query:", sample_q)
print("Sample neighbors:", custom_gt[sample_q][:10])

DB descriptors    : (39970, 128)
Query descriptors : (1000, 128)
DB global range   : 0 to 39999
Query global range: 40000 to 41000
Custom GT queries: 1000
Sample query: 40000
Sample neighbors: [1739, 19388, 33551, 25508, 7821, 37572, 37132, 36820, 34424, 15271]


In [25]:
sample_q = next(iter(custom_gt))
print("GT neighbors stored for sample query:", len(custom_gt[sample_q]))

GT neighbors stored for sample query: 500


In [26]:
# ============================================================
# CELL 4: Build PolyMP graphs from valid polygons only
# ============================================================

import numpy as np
import torch
from torch_geometric.data import Data

def normalize_polygon(coords):
    coords = coords - coords.mean(axis=0, keepdims=True)
    scale = np.linalg.norm(coords, axis=1).max()
    if scale > 0:
        coords = coords / scale
    return coords.astype(np.float32)

def build_features_and_edges_from_geom(geom):
    coords = np.asarray(geom.exterior.coords, dtype=np.float32)
    coords = coords[:-1] if len(coords) >= 2 and np.allclose(coords[0], coords[-1]) else coords
    coords = normalize_polygon(coords)

    N = len(coords)
    feats = []
    edges = []

    for i in range(N):
        p = coords[i]
        q = coords[(i + 1) % N]

        dx, dy = q - p
        edge_len = np.sqrt(dx * dx + dy * dy)
        angle = np.arctan2(dy, dx)

        feats.append([
            p[0], p[1],
            edge_len,
            angle,
            dx, dy
        ])

        edges.append([i, (i + 1) % N])
        edges.append([(i + 1) % N, i])

    x = np.asarray(feats, dtype=np.float32)
    edge_index = np.asarray(edges, dtype=np.int64).T
    return x, edge_index

graphs = []
graph_global_ids = []

for geom, gidx in zip(geometries_valid, valid_indices):
    x, edge_index = build_features_and_edges_from_geom(geom)
    graphs.append(
        Data(
            x=torch.tensor(x, dtype=torch.float32),
            edge_index=torch.tensor(edge_index, dtype=torch.long),
            poly_id=int(gidx)
        )
    )
    graph_global_ids.append(int(gidx))

global_to_graphrow = {gid: i for i, gid in enumerate(graph_global_ids)}

# DB / query graph rows aligned with original dataset split
db_graph_rows = [global_to_graphrow[int(gid)] for gid in db_global_ids if int(gid) in global_to_graphrow]
query_graph_rows = [global_to_graphrow[int(gid)] for gid in query_global_ids if int(gid) in global_to_graphrow]

db_graph_global_ids = [graph_global_ids[i] for i in db_graph_rows]
query_graph_global_ids = [graph_global_ids[i] for i in query_graph_rows]

print("DB graph rows   :", len(db_graph_rows))
print("Query graph rows:", len(query_graph_rows))

print("Graphs built:", len(graphs))
print("Sample graph:", graphs[0])
print("Sample global id:", graph_global_ids[0])

DB graph rows   : 39970
Query graph rows: 1000
Graphs built: 49963
Sample graph: Data(x=[56, 6], edge_index=[2, 112], poly_id=0)
Sample global id: 0


In [27]:
# ============================================================
# CELL: PolyMP model definition
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_mean
from torch_geometric.nn import global_mean_pool

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class PolyMPConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()

        # message function
        self.mlp_msg = nn.Sequential(
            nn.Linear(in_dim * 2, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )

        # update function
        self.mlp_update = nn.Sequential(
            nn.Linear(in_dim + out_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, x, edge_index):
        row, col = edge_index  # source, target

        # messages from neighbors
        m = torch.cat([x[row], x[col]], dim=1)   # [E, 2*F]
        m = self.mlp_msg(m)                      # [E, out_dim]

        # aggregate at target nodes
        agg = scatter_mean(m, col, dim=0, dim_size=x.size(0))

        # update node states
        out = self.mlp_update(torch.cat([x, agg], dim=1))
        return out


class PolygonEncoderPolyMP(nn.Module):
    def __init__(self, in_dim=6, hidden_dim=64, emb_dim=128, dropout=0.1):
        super().__init__()

        self.conv1 = PolyMPConv(in_dim, hidden_dim)
        self.conv2 = PolyMPConv(hidden_dim, hidden_dim)
        self.conv3 = PolyMPConv(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout)

        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, emb_dim),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)

        # graph-level embedding
        x = global_mean_pool(x, batch)

        x = self.proj(x)

        # normalize so cosine similarity = dot product
        x = F.normalize(x, dim=1)
        return x


model = PolygonEncoderPolyMP(
    in_dim=6,       # your graph node feature size
    hidden_dim=64,
    emb_dim=128,
    dropout=0.1
).to(DEVICE)

print(model)
print("Device:", DEVICE)

PolygonEncoderPolyMP(
  (conv1): PolyMPConv(
    (mlp_msg): Sequential(
      (0): Linear(in_features=12, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
    (mlp_update): Sequential(
      (0): Linear(in_features=70, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (conv2): PolyMPConv(
    (mlp_msg): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
    (mlp_update): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (conv3): PolyMPConv(
    (mlp_msg): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
    (mlp_update): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (dropout): Dropout

In [ ]:
# ============================================================
# NEW CELL: Train PolyMP with triplet loss from custom GT
# ============================================================

import random
import copy
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Batch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ------------------------------------------------------------
# Build positive pool from custom GT
# query gid -> positives from GT that also exist in graph space
# ------------------------------------------------------------
valid_graph_gid_set = set(graph_global_ids)
db_gid_set = set(db_graph_global_ids)

train_queries = []
positive_pool = {}

for q_gid, nn_list in custom_gt.items():
    if q_gid not in global_to_graphrow:
        continue

    pos = [gid for gid in nn_list[:20] if gid in db_gid_set and gid in global_to_graphrow]
    if len(pos) == 0:
        continue

    positive_pool[q_gid] = pos
    train_queries.append(q_gid)

print("Train queries with positives:", len(train_queries))


# ------------------------------------------------------------
# Negative sampling helper
# Harder than pure random: sample from DB but not from GT top-500
# ------------------------------------------------------------
gt_exclusion = {}
for q_gid, nn_list in custom_gt.items():
    gt_exclusion[q_gid] = set(nn_list[:500]) | {q_gid}

db_candidates_all = np.array(db_graph_global_ids, dtype=np.int32)

def sample_negative(q_gid):
    """
    Use clearly non-relevant negatives.
    Avoid top-100 GT neighbors entirely.
    """
    banned = set(custom_gt.get(q_gid, [])[:100]) | {q_gid}

    while True:
        neg_gid = int(np.random.choice(db_candidates_all))
        if neg_gid not in banned and neg_gid in global_to_graphrow:
            return neg_gid


# ------------------------------------------------------------
# Triplet dataset: (anchor graph, positive graph, negative graph)
# ------------------------------------------------------------
class PolyTripletDataset(Dataset):
    def __init__(self, query_gids, positive_pool, repeats_per_query=3):
        self.samples = []
        for q_gid in query_gids:
            pos_list = positive_pool[q_gid]
            for _ in range(repeats_per_query):
                p_gid = random.choice(pos_list)
                n_gid = sample_negative(q_gid)
                self.samples.append((q_gid, p_gid, n_gid))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        q_gid, p_gid, n_gid = self.samples[idx]
        a = graphs[global_to_graphrow[q_gid]]
        p = graphs[global_to_graphrow[p_gid]]
        n = graphs[global_to_graphrow[n_gid]]
        return a, p, n


def collate_triplets(batch):
    anchors = [x[0] for x in batch]
    positives = [x[1] for x in batch]
    negatives = [x[2] for x in batch]
    return anchors, positives, negatives


# ------------------------------------------------------------
# Create / reset model
# ------------------------------------------------------------
model = PolygonEncoderPolyMP(in_dim=6, hidden_dim=64, emb_dim=128, dropout=0.1).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = torch.nn.TripletMarginLoss(margin=0.2, p=2)

EPOCHS = 6
BATCH_SIZE = 64
REPEATS_PER_QUERY = 5

best_state = None
best_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    # rebuild triplets each epoch so negatives refresh
    train_ds = PolyTripletDataset(train_queries, positive_pool, repeats_per_query=REPEATS_PER_QUERY)
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_triplets
    )

    model.train()
    running_loss = 0.0
    num_batches = 0

    for anchors, positives, negatives in train_loader:
        optimizer.zero_grad()

        # batch each arm separately
        a_batch = Batch.from_data_list(anchors).to(DEVICE)
        p_batch = Batch.from_data_list(positives).to(DEVICE)
        n_batch = Batch.from_data_list(negatives).to(DEVICE)

        z_a = model(a_batch)
        z_p = model(p_batch)
        z_n = model(n_batch)

        # normalize before triplet loss
        z_a = F.normalize(z_a, p=2, dim=1)
        z_p = F.normalize(z_p, p=2, dim=1)
        z_n = F.normalize(z_n, p=2, dim=1)

        loss = criterion(z_a, z_p, z_n)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    epoch_loss = running_loss / max(num_batches, 1)
    print(f"Epoch {epoch}/{EPOCHS} | loss={epoch_loss:.4f}")

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        best_state = copy.deepcopy(model.state_dict())

# restore best model
if best_state is not None:
    model.load_state_dict(best_state)

print("Best training loss:", best_loss)

Train queries with positives: 1000
Epoch 1/12 | loss=0.3532
Epoch 2/12 | loss=0.3218
Epoch 3/12 | loss=0.3047
Epoch 4/12 | loss=0.3000
Epoch 5/12 | loss=0.2818
Epoch 6/12 | loss=0.2740
Epoch 7/12 | loss=0.2656
Epoch 8/12 | loss=0.2564
Epoch 9/12 | loss=0.2475
Epoch 10/12 | loss=0.2436
Epoch 11/12 | loss=0.2429
Epoch 12/12 | loss=0.2324
Best training loss: 0.23238503420428866


In [29]:
# ============================================================
# CELL 5: Generate embeddings from trained PolyMP
# ============================================================

import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

loader = DataLoader(graphs, batch_size=512, shuffle=False)

emb_list = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(DEVICE)
        h = model(batch)              # [B, 128]
        h = F.normalize(h, p=2, dim=1)
        emb_list.append(h.cpu())

H = torch.cat(emb_list, dim=0)

print("Embedding shape:", H.shape)

Embedding shape: torch.Size([49963, 128])


In [30]:
# ============================================================
# CELL 6: PolyMP Recall@10/50/100/500 vs custom GT (DB-only)
# ============================================================

import numpy as np

H_np = H.numpy()
H_db = H_np[db_graph_rows]

def get_topk_cosine_db_only(query_gid, k=10):
    if query_gid not in global_to_graphrow:
        return []

    q_idx = global_to_graphrow[query_gid]
    q_vec = H_np[q_idx]

    sims = H_db @ q_vec

    topk_local = np.argpartition(-sims, kth=k-1)[:k]
    topk_local = topk_local[np.argsort(-sims[topk_local])]

    return [db_graph_global_ids[i] for i in topk_local]

def recall_poly_multi(k_values=[10, 50, 100, 500], max_queries=200):
    queries = list(custom_gt.keys())
    if max_queries is not None:
        queries = queries[:max_queries]

    results = {}
    valid_counts = {}

    for k in k_values:
        scores = []

        for q in queries:
            if q not in custom_gt or len(custom_gt[q]) < k:
                continue

            pred = get_topk_cosine_db_only(q, k=k)
            if len(pred) < k:
                continue

            pred_set = set(pred)
            gt_set = set(custom_gt[q][:k])
            scores.append(len(pred_set & gt_set) / k)

        results[k] = float(np.mean(scores)) if scores else 0.0
        valid_counts[k] = len(scores)

    return results, valid_counts

K_VALUES = [10, 50, 100, 500]

recalls_poly, valid_counts_poly = recall_poly_multi(
    k_values=K_VALUES,
    max_queries=200
)

print("\n=== POLYMP RESULT ===")
for k in K_VALUES:
    print(f"PolyMP Recall@{k}: {recalls_poly[k]:.4f} | valid_queries={valid_counts_poly[k]}")


=== POLYMP RESULT ===
PolyMP Recall@10: 0.0240 | valid_queries=200
PolyMP Recall@50: 0.0350 | valid_queries=200
PolyMP Recall@100: 0.0378 | valid_queries=200
PolyMP Recall@500: 0.0625 | valid_queries=200
